In [2]:
import pygame              # Pygame kütüphanesini içe aktarır
import sys                 # Sistem işlemleri için sys modülünü içe aktarır
import random              # Rastgele seçimler için random modülü
import numpy as np         # Matematiksel işlemler ve Q-table için numpy
import os                  # Dosya kontrol işlemleri için os modülü

pygame.init()  # Pygame başlatılıyor

# ---------------------------------------
# GRID AYARLARI
# ---------------------------------------
GRID = 6             # Grid boyutu (6x6)
CELL = 100           # Her hücrenin piksel boyutu
SCREEN_W = GRID * CELL + 600  # Q-table görseli için ekstra genişlik
SCREEN_H = GRID * CELL        # Ekran yüksekliği
screen = pygame.display.set_mode((SCREEN_W, SCREEN_H))  # Pygame ekranı oluşturulur
pygame.display.set_caption("Taxi Q-Learning Simulation")  # Pencere başlığı ayarlanır
clock = pygame.time.Clock()  # FPS kontrolü için clock nesnesi oluşturulur

# ---------------------------------------
# DURAKLAR
# ---------------------------------------
stops = [
    (0, 0, "A"),  # Üst sol köşe
    (0, 5, "B"),  # Üst sağ köşe
    (5, 0, "C"),  # Alt sol köşe
    (5, 5, "D")   # Alt sağ köşe
]

# ---------------------------------------
# ENGELLER
# ---------------------------------------
walls = [
    (1, 1, "RIGHT"),  # (satır, sütun, yön)
    (2, 2, "DOWN"),
    (3, 3, "UP"),
    (4, 1, "LEFT"),
]

# ---------------------------------------
# Q-LEARNING PARAMETRELER
# ---------------------------------------
alpha = 0.1                 # Öğrenme hızı
gamma = 0.95                # İndirim faktörü
epsilon = 0.1               # Keşfetme oranı
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]  # Aksiyon hareketleri (yukarı, aşağı, sol, sağ)
ACTION_COUNT = 4            # Aksiyon sayısı
Q = np.zeros((GRID, GRID, 4, 4, ACTION_COUNT))  # Q-table sıfırlarla başlatılır
if os.path.exists("qtable.npy"):     # qtable dosyası var mı kontrol edilir
    Q = np.load("qtable.npy")        # Q-tablosu dosyadan yüklenir
episode_count = 0            # Episode sayacı
episode_finished = True      # Başlangıçta yeni episode bekleniyor

# ---------------------------------------
# ENGEL KONTROL FONKSİYONLARI
# ---------------------------------------
def wall_exists(r,c,direction):
    return (r,c,direction) in walls  # Belirtilen koordinatta o yönde duvar olup olmadığını kontrol eder

def can_move(r,c,action):
    dr,dc = ACTIONS[action]          # Aksiyonun satır/sütun değişimleri alınır
    nr,nc = r+dr,c+dc                # Yeni pozisyon hesaplanır

    if nr<0 or nr>=GRID or nc<0 or nc>=GRID:  # Grid dışına çıkma durumu
        return False

    # Hareket yönünde duvar olup olmadığını kontrol eder
    if action==0 and wall_exists(r,c,"UP"): return False
    if action==1 and wall_exists(r,c,"DOWN"): return False
    if action==2 and wall_exists(r,c,"LEFT"): return False
    if action==3 and wall_exists(r,c,"RIGHT"): return False

    # Hedef hücrenin ters yönündeki duvarı kontrol eder
    if action==0 and wall_exists(nr,nc,"DOWN"): return False
    if action==1 and wall_exists(nr,nc,"UP"): return False
    if action==2 and wall_exists(nr,nc,"RIGHT"): return False
    if action==3 and wall_exists(nr,nc,"LEFT"): return False

    return True  # Hareket edilebilir

# ---------------------------------------
# GRAFİK FONKSİYONLARI
# ---------------------------------------
def draw_grid():
    for r in range(GRID):            # Tüm satırlarda döner
        for c in range(GRID):        # Tüm sütunlarda döner
            rect = pygame.Rect(c*CELL,r*CELL,CELL,CELL)  # Hücrenin ekran dikdörtgenini oluşturur
            pygame.draw.rect(screen,(120,120,120),rect)   # Hücreyi renkle doldurur
            pygame.draw.rect(screen,(180,180,180),rect,2) # Hücre sınırını çizer

def draw_walls():
    thick=10               # Duvar çizim kalınlığı
    color=(30,30,30)       # Duvar çizim rengi
    for r,c,d in walls:    # Tüm duvarları dolaşır
        x=c*CELL           # Duvarın x pozisyonu hesaplanır
        y=r*CELL           # Duvarın y pozisyonu hesaplanır
        # Duvar yönüne göre çizim yapılır
        if d=="UP": pygame.draw.line(screen,color,(x,y),(x+CELL,y),thick)
        if d=="DOWN": pygame.draw.line(screen,color,(x,y+CELL),(x+CELL,y+CELL),thick)
        if d=="LEFT": pygame.draw.line(screen,color,(x,y),(x,y+CELL),thick)
        if d=="RIGHT": pygame.draw.line(screen,color,(x+CELL,y),(x+CELL,y+CELL),thick)

def draw_stops():
    font = pygame.font.SysFont("Arial",40,bold=True)  # Durak yazı fontu
    colors = [(200,80,80),(80,200,80),(80,80,200),(200,200,80)]  # Durak renkleri
    for i,(r,c,label) in enumerate(stops):                       # Durakları çiz
        rect=pygame.Rect(c*CELL+15,r*CELL+15,CELL-30,CELL-30)   # Durak dikdörtgeni
        pygame.draw.rect(screen,colors[i],rect,border_radius=15) # Durak arka planını çiz
        text=font.render(label,True,(0,0,0))                    # Durak etiketi yazısı
        screen.blit(text,text.get_rect(center=rect.center))     # Etiketi merkeze yerleştirir

def draw_taxi(r,c):
    rect=pygame.Rect(c*CELL+25,r*CELL+25,CELL-50,CELL-50)  # Taxi için dikdörtgen oluşturur
    pygame.draw.rect(screen,(255,255,255),rect,border_radius=10)  # Taksiyi çizer

def draw_passenger(r,c):
    center=(c*CELL+CELL//2,r*CELL+CELL//2)           # Yolcu merkez koordinatı
    pygame.draw.circle(screen,(0,0,0),center,22)     # Dış daire
    pygame.draw.circle(screen,(230,230,230),center,18)  # İç daire
    pygame.draw.circle(screen,(0,0,0),(center[0],center[1]-8),5)  # Baş
    pygame.draw.line(screen,(0,0,0),(center[0],center[1]-3),(center[0],center[1]+10),3) # Gövde
    pygame.draw.line(screen,(0,0,0),(center[0],center[1]+10),(center[0]-7,center[1]+20),3) # Sol bacak
    pygame.draw.line(screen,(0,0,0),(center[0],center[1]+10),(center[0]+7,center[1]+20),3) # Sağ bacak

def draw_passenger_goal(dest_idx):
    r,c,_=stops[dest_idx]                                 # Hedef durağın koordinatlarını alır
    rect=pygame.Rect(c*CELL+40,r*CELL+40,20,20)           # Kırmızı hedef işareti
    pygame.draw.rect(screen,(255,0,0),rect)               # Hedef kareyi çizer

# ---------------------------------------
# Q-TABLE GÖRSELLEŞTİRME
# ---------------------------------------
def draw_qtable_grid_edges(pass_idx,dest_idx):
    font = pygame.font.SysFont("Arial", 20)               # Q değerleri yazı fontu
    x0 = GRID * CELL + 20    # Q-table başlangıç x pozisyonu
    y0 = 20                  # Q-table başlangıç y pozisyonu
    cell_size = 80           # Q-table hücre boyutu

    for r in range(GRID):
        for c in range(GRID):
            qvals = Q[r, c, pass_idx, dest_idx]           # Hücrenin Q değerleri alınır
            qvals_rounded = np.round(qvals, 1)            # Görsel için yuvarlanır
            cell_x = x0 + c*(cell_size + 15)              # Hücrenin x pozisyonu
            cell_y = y0 + r*(cell_size + 15)              # Hücrenin y pozisyonu

            # Oklar ve Q değerleri ekrana çizilir
            screen.blit(font.render("▲"+str(qvals_rounded[0]), True, (255,255,255)),
                        (cell_x + cell_size//2 - 10, cell_y))

            screen.blit(font.render("▼"+str(qvals_rounded[1]), True, (255,255,255)),
                        (cell_x + cell_size//2 - 10, cell_y + cell_size - 25))

            screen.blit(font.render("◄"+str(qvals_rounded[2]), True, (255,255,255)),
                        (cell_x, cell_y + cell_size//2 - 10))

            screen.blit(font.render("►"+str(qvals_rounded[3]), True, (255,255,255)),
                        (cell_x + cell_size - 35, cell_y + cell_size//2 - 10))

# ---------------------------------------
# Q-LEARNING FONKSİYONLARI
# ---------------------------------------
def choose_action(r,c,p,d):
    if random.random()<epsilon:         # Epsilon-greedy keşif kontrolü
        return random.randint(0,ACTION_COUNT-1)  # Rastgele aksiyon seç
    return np.argmax(Q[r,c,p,d])        # En yüksek Q değerine sahip aksiyonu seç

def reset_episode():
    taxi_r=random.randint(0,GRID-1)     # Taksinin başlangıç satırı
    taxi_c=random.randint(0,GRID-1)     # Taksinin başlangıç sütunu
    passenger_idx=random.randint(0,3)   # Yolcunun durak indexi
    destination_idx=random.randint(0,3) # Hedef durak indexi
    while destination_idx==passenger_idx:  # Yolcu hedefi aynı olamaz
        destination_idx=random.randint(0,3)
    onboard=False                       # Yolcu takside değil
    return taxi_r,taxi_c,passenger_idx,destination_idx,onboard

# ---------------------------------------
# ANA DÖNGÜ
# ---------------------------------------
taxi_r=taxi_c=0      # Taksi başlangıç konumu
pass_idx=dest_idx=0  # Başlangıç yolcu ve hedef indexi
onboard=False        # Başlangıçta yolcu araçta değil

while True:
    for event in pygame.event.get():    # Pygame event döngüsü
        if event.type==pygame.QUIT:
            np.save("qtable.npy",Q)     # Kapanışta Q-table kaydedilir
            pygame.quit()               # Pygame kapatılır
            sys.exit()                  # Program sonlandırılır

        if event.type==pygame.KEYDOWN:
            if event.key==pygame.K_SPACE and episode_finished:
                taxi_r,taxi_c,pass_idx,dest_idx,onboard=reset_episode()  # Yeni episode başlat
                episode_finished=False
                episode_count+=1         # Episode sayısını artır

    screen.fill((50,50,50))             # Arka planı doldur
    draw_grid()                         # Grid çiz
    draw_walls()                        # Duvarları çiz
    draw_stops()                        # Durakları çiz
    draw_taxi(taxi_r,taxi_c)            # Taksiyi çiz
    draw_passenger_goal(dest_idx)       # Hedef durağı işaretle

    if not onboard:
        pr,pc,_=stops[pass_idx]         # Yolcunun konumunu al
        draw_passenger(pr,pc)           # Yolcuyu çiz

    draw_qtable_grid_edges(pass_idx,dest_idx)  # Q-table görselini çiz

    if episode_finished:
        font=pygame.font.SysFont("Arial",60,bold=True)  # Başlatma teks fontu
        text=font.render("PRESS SPACE",True,(255,255,255))  # Başlatma yazısı
        screen.blit(text,(80,SCREEN_H//2-30))              # Yazıyı ekrana yerleştir
        pygame.display.flip()                               # Ekranı güncelle
        clock.tick(60)                                      # FPS
        continue

    old_r,old_c=taxi_r,taxi_c            # Güncel taksi pozisyonunu sakla
    action=choose_action(taxi_r,taxi_c,pass_idx,dest_idx)  # Aksiyon seç

    if can_move(taxi_r,taxi_c,action):   # Hareket mümkün mü kontrol et
        dr,dc=ACTIONS[action]            # Aksiyonun hareket yönü
        taxi_r+=dr                       # Taksi satırını güncelle
        taxi_c+=dc                       # Taksi sütununu güncelle

    reward=-1                            # Varsayılan adım cezası

    if not onboard:                      # Yolcu henüz alınmamışsa
        pr,pc,_=stops[pass_idx]          # Yolcunun konumu
        if taxi_r==pr and taxi_c==pc:    # Taksi yolcuya ulaştı mı?
            onboard=True
            reward=20                    # Yolcu alınca ödül

    else:                                # Yolcu araçtaysa
        dr,dc,_=stops[dest_idx]          # Hedef konumu
        if taxi_r==dr and taxi_c==dc:    # Teslim edildiyse
            reward=50                    # Teslim ödülü
            episode_finished=True

    best_next=np.max(Q[taxi_r,taxi_c,pass_idx,dest_idx])  # Sonraki durumun en iyi Q değeri

    Q[old_r,old_c,pass_idx,dest_idx,action]+=alpha*(reward+gamma*best_next-Q[old_r,old_c,pass_idx,dest_idx,action])  # Q-learning güncellemesi

    pygame.display.flip()               # Ekranı güncelle
    clock.tick(5)                       # FPS sınırlaması

    if episode_finished:
        np.save("qtable.npy",Q)         # Episode sonunda Q-table kaydedilir


SystemExit: 

C:\Users\Public\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
